In [3]:
import astropy.units as u
import astropy.constants as c
from astropy.coordinates import SkyCoord, search_around_sky
from astropy.time import Time
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import glob
from astropy.io import fits
from itertools import combinations
import pickle
from astropy.cosmology import WMAP9 as cosmo
import seaborn as sns
from astropy.table import Table



import lime
from pathlib import Path
import pprint

lime.show_instrument_cfg()


Single ".fits" spectra files configuration:
0 nirspec) 	 units_wave: um, units_flux: MJy, pixel_mask: nan, res_power: None
1 isis) 	 units_wave: Angstrom, units_flux: FLAM, pixel_mask: nan, res_power: None
2 osiris) 	 units_wave: Angstrom, units_flux: FLAM, pixel_mask: nan, res_power: None
3 sdss) 	 units_wave: Angstrom, units_flux: 1e-17*FLAM, pixel_mask: nan, res_power: None
4 desi) 	 units_wave: Angstrom, units_flux: 1e-17*FLAM, pixel_mask: nan, res_power: None

Cube ".fits" spectra files configuration:
0 manga) 	 units_wave: Angstrom, units_flux: 1e-17*FLAM,pixel_mask: nan, res_power: None
1 muse) 	 units_wave: Angstrom, units_flux: 1e-20*FLAM,pixel_mask: nan, res_power: None
2 megara) 	 units_wave: Angstrom, units_flux: Jy,pixel_mask: nan, res_power: None
3 miri) 	 units_wave: um, units_flux: MJy,pixel_mask: nan, res_power: None


In [4]:
#bands_df = lime.line_bands(vacuum=True)
#bands_df

plt.close('all')

In [5]:
obs_cfg = lime.load_cfg('config.toml')
pprint.pprint(obs_cfg)

LiMe_Error: The configuration file was not found at: config.toml

In [ ]:
#Should do this for each spectra in a list of spectra
for i in range(len(CAPERS_prism_FLUX)):
    #Create the LiME object from a spectra 
    obj = lime.Spectrum(input_wave = CAPERS_prism_WAV[i], input_flux = CAPERS_prism_FLUX[i], 
                    input_err=CAPERS_prism_FLUX_e[i],
                    redshift=CAPERS_prism_z[i], units_wave = 'um', units_flux = 'Jy')
    #Get the object identifyer
    idi = CAPERS_prism_mptids[i]
    print(idi)
    #Unit conversions
    obj.unit_conversion('AA', 'FLAM')
    obj.plot.spectrum(rest_frame=True)
    #Create a bands file path, populate it for this object
    bands_file = f'{idi}_PRbands.txt'
    bands_clean_df = lime.line_bands(wave_intvl=obj.wave.data, redshift=obj.redshift,
                                ref_bands='CAPERS_prism_bands_redshift.txt')
    #Save the bands file
    lime.save_frame(bands_file, bands_clean_df)
    obj.fit.continuum(degree_list=[3, 6, 6], emis_threshold=[3, 2, 1.5], plot_steps=True)
    
    
    try:
        matched_bands = obj.line_detection(bands_file, sigma_threshold=3, plot_steps=True)
        obj.plot.spectrum(label='matched lines', bands=matched_bands, log_scale=True)
        obj.fit.frame(bands=matched_bands, fit_conf=obs_cfg, id_conf_prefix=idi)
        print(f'# Default line fitting')
       # pprint.pprint(obs_cfg['default_line_fitting'])
        print(f'\n# Individual line fitting')
        #pprint.pprint(obs_cfg['gp121903_line_fitting'])
        #pprint.pprint({**obs_cfg['default_line_fitting'], **obs_cfg['gp121903_line_fitting']})
        # Display the fits on the spectrum
        obj.plot.spectrum(include_fits=True, rest_frame=True)
        # Display a grid with the fits
        obj.plot.grid()     
        obj.plot.spectrum(bands=bands_clean_df)
        #The lines are shaded with the yellow bars on the correct region, but no fits are performed
        obj.check.bands(bands_clean_df, ref_bands=lime.line_bands(wave_intvl=obj.wave))
        #Nothing saves here
        obj.save_frame(f'/home/kelcey/eelg_properties/bands/{idi}_CAPERS_fluxes_table.txt')
        plt.close('all')
    except:
        print('FAILED')
    


In [ ]:
#Should do this for each spectra in a list of spectra
for i in range(len(RUB_prism_FLUX)):
    #Create the LiME object from a spectra 
    obj = lime.Spectrum(input_wave = RUB_prism_WAV[i], input_flux = RUB_prism_FLUX[i], 
                    input_err=RUB_prim_FLUX_e[i],
                    redshift=RUB_prism_z[i], units_wave = 'um', units_flux = 'Jy')
    #Get the object identifyer
    idi = RUB_prism_mptid[i]
    print(idi)
    #Unit conversions
    obj.unit_conversion('AA', 'FLAM')
    obj.plot.spectrum(rest_frame=True)
    #Create a bands file path, populate it for this object
    bands_file = f'{idi}_PRbands.txt'
    bands_clean_df = lime.line_bands(wave_intvl=obj.wave.data, redshift=obj.redshift,
                                ref_bands='CAPERS_prism_bands_redshift.txt')
    #Save the bands file
    lime.save_frame(bands_file, bands_clean_df)
    obj.fit.continuum(degree_list=[3, 6, 6], emis_threshold=[3, 2, 1.5], plot_steps=True)
    
    
    try:
        matched_bands = obj.line_detection(bands_file, sigma_threshold=3, plot_steps=True)
        obj.plot.spectrum(label='matched lines', bands=matched_bands, log_scale=True)
        obj.fit.frame(bands=matched_bands, fit_conf=obs_cfg, id_conf_prefix=idi)
        print(f'# Default line fitting')
       # pprint.pprint(obs_cfg['default_line_fitting'])
        print(f'\n# Individual line fitting')
        #pprint.pprint(obs_cfg['gp121903_line_fitting'])
        #pprint.pprint({**obs_cfg['default_line_fitting'], **obs_cfg['gp121903_line_fitting']})
        # Display the fits on the spectrum
        obj.plot.spectrum(include_fits=True, rest_frame=True)
        # Display a grid with the fits
        obj.plot.grid()     
        obj.plot.spectrum(bands=bands_clean_df)
        #The lines are shaded with the yellow bars on the correct region, but no fits are performed
        obj.check.bands(bands_clean_df, ref_bands=lime.line_bands(wave_intvl=obj.wave))
        #Nothing saves here
        obj.save_frame(f'/home/kelcey/eelg_properties/bands/{idi}_PRISMRUBIES_fluxes_table.txt')
        plt.close('all')
    except:
        print('FAILED')
    


In [ ]:
#Should do this for each spectra in a list of spectra
for i in range(len(RUB_grat_FLUX)):
    #Create the LiME object from a spectra 
    obj = lime.Spectrum(input_wave = RUB_grat_WAV[i], input_flux = RUB_grat_FLUX[i], 
                    input_err=RUB_grat_FLUX_e[i],
                    redshift=RUB_grat_z[i], units_wave = 'um', units_flux = 'Jy')
    #Get the object identifyer
    idi = RUB_grat_mptids[i]
    print(idi)
    #Unit conversions
    obj.unit_conversion('AA', 'FLAM')
    obj.plot.spectrum(rest_frame=True)
    #Create a bands file path, populate it for this object
    bands_file = f'{idi}GR_bands.txt'
    bands_clean_df = lime.line_bands(wave_intvl=obj.wave.data, redshift=obj.redshift,
                                ref_bands='CAPERS_prism_bands_redshift.txt')
    #Save the bands file
    lime.save_frame(bands_file, bands_clean_df)
    obj.fit.continuum(degree_list=[3, 6, 6], emis_threshold=[3, 2, 1.5], plot_steps=True)
    
    
    try:
        matched_bands = obj.line_detection(bands_file, sigma_threshold=3, plot_steps=True)
        obj.plot.spectrum(label='matched lines', bands=matched_bands, log_scale=True)
        obj.fit.frame(bands=matched_bands, id_conf_prefix=idi)
        print(f'# Default line fitting')
       # pprint.pprint(obs_cfg['default_line_fitting'])
        print(f'\n# Individual line fitting')
        #pprint.pprint(obs_cfg['gp121903_line_fitting'])
        #pprint.pprint({**obs_cfg['default_line_fitting'], **obs_cfg['gp121903_line_fitting']})
        # Display the fits on the spectrum
        obj.plot.spectrum(include_fits=True, rest_frame=True)
        # Display a grid with the fits
        obj.plot.grid()     
        obj.plot.spectrum(bands=bands_clean_df)
        #The lines are shaded with the yellow bars on the correct region, but no fits are performed
        obj.check.bands(bands_clean_df, ref_bands=lime.line_bands(wave_intvl=obj.wave))
        #Nothing saves here
        obj.save_frame(f'/home/kelcey/eelg_properties/bands/{idi}_GRATRUBIES_fluxes_table.txt')
        plt.close('all')
    except:
        print('FAILED')
    


In [ ]:
#Should do this for each spectra in a list of spectra
for i in range(len(CEERS_prism_FLUX)):
    #Create the LiME object from a spectra 
    obj = lime.Spectrum(input_wave = CEERS_prism_WAV[i], input_flux = CEERS_prism_FLUX[i], 
                    input_err=CEERS_prism_FLUX_e[i],
                    redshift=CEERS_prism_z[i], units_wave = 'um', units_flux = 'Jy')
    #Get the object identifyer
    idi = CEERS_prism_mptids[i]
    print(idi)
    #Unit conversions
    obj.unit_conversion('AA', 'FLAM')
    obj.plot.spectrum(rest_frame=True)
    #Create a bands file path, populate it for this object
    bands_file = f'{idi}_bands.txt'
    bands_clean_df = lime.line_bands(wave_intvl=obj.wave.data, redshift=obj.redshift,
                                ref_bands='CAPERS_prism_bands_redshift.txt')
    #Save the bands file
    lime.save_frame(bands_file, bands_clean_df)
    obj.fit.continuum(degree_list=[3, 6, 6], emis_threshold=[3, 2, 1.5], plot_steps=True)
    
    
    try:
        matched_bands = obj.line_detection(bands_file, sigma_threshold=3, plot_steps=True)
        obj.plot.spectrum(label='matched lines', bands=matched_bands)#, log_scale=True)
        obj.fit.frame(bands=matched_bands, id_conf_prefix=idi)
        print(f'# Default line fitting')
       # pprint.pprint(obs_cfg['default_line_fitting'])
        print(f'\n# Individual line fitting')
        #pprint.pprint(obs_cfg['gp121903_line_fitting'])
        #pprint.pprint({**obs_cfg['default_line_fitting'], **obs_cfg['gp121903_line_fitting']})
        # Display the fits on the spectrum
        obj.plot.spectrum(include_fits=True, rest_frame=True)
        # Display a grid with the fits
        obj.plot.grid()     
        obj.plot.spectrum(bands=bands_clean_df)
        #The lines are shaded with the yellow bars on the correct region, but no fits are performed
        obj.check.bands(bands_clean_df, ref_bands=lime.line_bands(wave_intvl=obj.wave))
        #Nothing saves here
        obj.save_frame(f'/home/kelcey/eelg_properties/bands/{idi}_PRISM_fluxes_table.txt')
        plt.close('all')
    except:
        print('FAILED')
    


In [ ]:
#Should do this for each spectra in a list of spectra
for i in range(len(CEERS_grat_FLUX)):
    #Create the LiME object from a spectra 
    obj = lime.Spectrum(input_wave = np.array(CEERS_grat_WAV[i]), input_flux = np.array(CEERS_grat_FLUX[i]), 
                    input_err=np.array(CEERS_grat_FLUX_e[i]),
                    redshift=CEERS_grat_z[i], units_wave = 'um', units_flux = 'Jy')
    #Get the object identifyer
    idi = CEERS_grat_mptids[i]
    print(idi)
    #Unit conversions
    obj.unit_conversion('AA', 'FLAM')
    obj.plot.spectrum(rest_frame=True)
    #Create a bands file path, populate it for this object
    bands_file = f'{idi}_bands.txt'
    bands_clean_df = lime.line_bands(wave_intvl=obj.wave.data, redshift=obj.redshift,
                                ref_bands='CAPERS_prism_bands_redshift.txt')
    #Save the bands file
    lime.save_frame(bands_file, bands_clean_df)
    obj.fit.continuum(degree_list=[3, 6, 6], emis_threshold=[3, 2, 1.5], plot_steps=True)
    
    
    try:
        matched_bands = obj.line_detection(bands_file, sigma_threshold=3, plot_steps=True)
        obj.plot.spectrum(label='matched lines', bands=matched_bands, log_scale=True)
        obj.fit.frame(bands=matched_bands, id_conf_prefix=idi)
        print(f'# Default line fitting')
       # pprint.pprint(obs_cfg['default_line_fitting'])
        print(f'\n# Individual line fitting')
        #pprint.pprint(obs_cfg['gp121903_line_fitting'])
        #pprint.pprint({**obs_cfg['default_line_fitting'], **obs_cfg['gp121903_line_fitting']})
        # Display the fits on the spectrum
        obj.plot.spectrum(include_fits=True, rest_frame=True)
        # Display a grid with the fits
        obj.plot.grid()


        # Save the data
        #gp_spec.save_frame('../sample_data/example3_linelog.fits', page='GP121903_a')

        #lime.save_frame('../sample_data/example3_linelog.fits', gp_spec.frame,  page='GP121903b')



        #Go through each of the lines specified in the bands file, force LiME to keep running
        #if it cannot fit a line
        #line is just a string identifier, for example 'H1_1216A'

        #for line in bands_clean_df.index.values:
        #    try:
        #        obj.fit.frame(bands_clean_df.loc[line])
        #    except Exception as e:
        #        print(f"Could not fit {line}: {e}")
         #The spectrum plots ok       
        obj.plot.spectrum(bands=bands_clean_df)
        #The lines are shaded with the yellow bars on the correct region, but no fits are performed
        obj.check.bands(bands_clean_df, ref_bands=lime.line_bands(wave_intvl=obj.wave))
        plt.close('all')
        #Nothing saves here
        obj.save_frame(f'/home/kelcey/eelg_properties/bands/{idi}_GRAT_fluxes_table.txt')
    except:
        print('FAILED')
    
